In [1]:
# Project 1: Recruitment Funnel - create synthetic (pretend) data
# Based on a real 8-stage care-sector recruitment process

import os
import numpy as np      # numpy = maths tools
import pandas as pd     # pandas = Excel inside Python

rng = np.random.default_rng(42)   # 42 = fixed "seed", so we get the same data every time
n_candidates = 1200

stages = ["applied", "documents_received", "interviewed", "offer_made",
          "training_completed", "induction_attended", "shadowing_completed", "started"]

# chance of passing each stage
pass_rate = {"documents_received": 0.70, "interviewed": 0.65, "offer_made": 0.75,
             "training_completed": 0.80, "induction_attended": 0.92,
             "shadowing_completed": 0.88, "started": 0.95}

# typical days each stage takes
typical_days = {"documents_received": 3, "interviewed": 5, "offer_made": 2,
                "training_completed": 14, "induction_attended": 4,
                "shadowing_completed": 21, "started": 3}

# some sources bring stronger candidates than others
source_boost = {"Indeed": 0.00, "Website": 0.02, "Referral": 0.08,
                "Facebook": -0.06, "Agency": 0.03}

rows = []
for i in range(n_candidates):
    source = rng.choice(list(source_boost), p=[0.40, 0.15, 0.15, 0.20, 0.10])
    role = rng.choice(["Healthcare Assistant", "Domiciliary Care Worker"], p=[0.45, 0.55])
    applied = pd.Timestamp("2025-01-01") + pd.Timedelta(days=int(rng.integers(0, 540)))
    row = {"candidate_id": f"C{i+1:04d}", "role": role, "source": source, "applied_date": applied}
    current = applied
    for stage in stages[1:]:
        days = max(1, int(rng.gamma(2.0, typical_days[stage] / 2.0)))
        chance = pass_rate[stage] + source_boost[source]
        if stage == "training_completed" and days > 18:   # slow training = more drop-out
            chance -= 0.25
        if rng.random() > chance:   # candidate drops out here
            break
        current = current + pd.Timedelta(days=days)
        row[f"{stage}_date"] = current
    rows.append(row)

df = pd.DataFrame(rows)
df = df.reindex(columns=["candidate_id", "role", "source"] + [f"{s}_date" for s in stages])

# add realistic mess (like real HR trackers)
messy = rng.choice(df.index, size=40, replace=False)
df.loc[messy[:20], "source"] = df.loc[messy[:20], "source"].str.lower()   # "indeed" vs "Indeed"
df.loc[messy[20:30], "source"] = df.loc[messy[20:30], "source"] + " "     # extra spaces
df.loc[messy[30:], "role"] = np.nan                                       # missing roles
df = pd.concat([df, df.sample(15, random_state=1)], ignore_index=True)    # duplicate rows

os.makedirs("data", exist_ok=True)
df.to_csv("data/recruitment_raw.csv", index=False)

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (1215, 11)


,candidate_id,role,source,applied_date,documents_received_date,interviewed_date,offer_made_date,training_completed_date,induction_attended_date,shadowing_completed_date,started_date
0,C0001,Healthcare Assistant,Facebook,2025-08-22,NaT,NaT,NaT,NaT,NaT,NaT,NaT
1,C0002,Domiciliary Care Worker,Facebook,2026-04-09,2026-04-11,NaT,NaT,NaT,NaT,NaT,NaT
2,C0003,Healthcare Assistant,Website,2025-02-19,2025-02-20,NaT,NaT,NaT,NaT,NaT,NaT
3,C0004,Domiciliary Care Worker,Facebook,2025-10-27,2025-10-28,NaT,NaT,NaT,NaT,NaT,NaT
4,C0005,Healthcare Assistant,Agency,2026-05-04,2026-05-07,2026-05-09,2026-05-11,2026-05-28,2026-05-31,2026-06-20,2026-06-23
